In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn.functional as F
from src.activations import relu, squared_relu, cubic_relu
from src.base_functions import relu_H5, squared_relu_H5, cubic_relu_H5
from src.fractal_functions import alpha_fractalize
from src.fractal_activation import FractalActivation

In [2]:
# Precompute fractal LUTs (same params as training)
a, b = -1, 1
n_subintervals = 6
n_iter = 2
alpha = [0, 0, 0, 0.01, 0.02, 0.03]

relu_fractal = alpha_fractalize(relu, lambda z: relu_H5(z, x1=a, xN=b), a, b, n_subintervals, alpha, n_iter, True)
squared_relu_fractal = alpha_fractalize(squared_relu, lambda z: squared_relu_H5(z, x1=a, xN=b), a, b, n_subintervals, alpha, n_iter, True)
cubic_relu_fractal = alpha_fractalize(cubic_relu, lambda z: cubic_relu_H5(z, x1=a, xN=b), a, b, n_subintervals, alpha, n_iter, True)

# Create FractalActivation modules
f_relu = FractalActivation(relu_fractal, lambda x: F.relu(x))
f_squared_relu = FractalActivation(squared_relu_fractal, lambda x: F.relu(x) ** 2)
f_cubic_relu = FractalActivation(cubic_relu_fractal, lambda x: F.relu(x) ** 3)

f(-1) = 0.0, f(1) = 1.0
g(-1) = 0.0, g(1) = 1.0
f(-1) = 0.0, f(1) = 1.0
g(-1) = 0.0, g(1) = 1.0
f(-1) = 0.0, f(1) = 1.0
g(-1) = 0.0, g(1) = 1.0


In [4]:
# Test values: -4 (outside, left), 6 (outside, right), 0.5 (inside)
test_values = [-300.0, 5.0, 100]
x = torch.tensor(test_values)

print("=" * 70)
print(f"{'Input':>8}  |  {'Fractal Act':>12}  |  {'Classical':>12}  |  {'Match?':>8}  |  Zone")
print("=" * 70)

# --- f_relu ---
print("\n--- Fractal ReLU ---")
fractal_out = f_relu(x)
classical_out = F.relu(x)
for i, v in enumerate(test_values):
    zone = 'INSIDE [-1,1]' if -1 <= v <= 1 else 'OUTSIDE'
    match = '✓' if torch.isclose(fractal_out[i], classical_out[i], atol=1e-4) else '✗ (fractal)'
    if -1 <= v <= 1:
        match = '— (fractal)'
    print(f"{v:>8.1f}  |  {fractal_out[i].item():>12.6f}  |  {classical_out[i].item():>12.6f}  |  {match:>11}  |  {zone}")

# --- f_squared_relu ---
print("\n--- Fractal Squared ReLU ---")
fractal_out = f_squared_relu(x)
classical_out = F.relu(x) ** 2
for i, v in enumerate(test_values):
    zone = 'INSIDE [-1,1]' if -1 <= v <= 1 else 'OUTSIDE'
    match = '✓' if torch.isclose(fractal_out[i], classical_out[i], atol=1e-4) else '✗ (fractal)'
    if -1 <= v <= 1:
        match = '— (fractal)'
    print(f"{v:>8.1f}  |  {fractal_out[i].item():>12.6f}  |  {classical_out[i].item():>12.6f}  |  {match:>11}  |  {zone}")

# --- f_cubic_relu ---
print("\n--- Fractal Cubic ReLU ---")
fractal_out = f_cubic_relu(x)
classical_out = F.relu(x) ** 3
for i, v in enumerate(test_values):
    zone = 'INSIDE [-1,1]' if -1 <= v <= 1 else 'OUTSIDE'
    match = '✓' if torch.isclose(fractal_out[i], classical_out[i], atol=1e-4) else '✗ (fractal)'
    if -1 <= v <= 1:
        match = '— (fractal)'
    print(f"{v:>8.1f}  |  {fractal_out[i].item():>12.6f}  |  {classical_out[i].item():>12.6f}  |  {match:>11}  |  {zone}")

print("\n" + "=" * 70)
print("Expected: OUTSIDE values should MATCH classical exactly.")
print("          INSIDE values use fractal interpolation (may differ).")

   Input  |   Fractal Act  |     Classical  |    Match?  |  Zone

--- Fractal ReLU ---
  -300.0  |      0.000000  |      0.000000  |            ✓  |  OUTSIDE
     5.0  |      5.000000  |      5.000000  |            ✓  |  OUTSIDE
   100.0  |    100.000000  |    100.000000  |            ✓  |  OUTSIDE

--- Fractal Squared ReLU ---
  -300.0  |      0.000000  |      0.000000  |            ✓  |  OUTSIDE
     5.0  |     25.000000  |     25.000000  |            ✓  |  OUTSIDE
   100.0  |  10000.000000  |  10000.000000  |            ✓  |  OUTSIDE

--- Fractal Cubic ReLU ---
  -300.0  |      0.000000  |      0.000000  |            ✓  |  OUTSIDE
     5.0  |    125.000000  |    125.000000  |            ✓  |  OUTSIDE
   100.0  |  1000000.000000  |  1000000.000000  |            ✓  |  OUTSIDE

Expected: OUTSIDE values should MATCH classical exactly.
          INSIDE values use fractal interpolation (may differ).
